<a href="https://colab.research.google.com/github/remix0091/Big-Data/blob/main/L3_Reports_with_Apache_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Lab3-Programming-Languages-Report")
    .getOrCreate()
)

In [4]:
# Чтение данных
posts_path = "posts_sample.xml"
languages_path = "programming-languages.csv"

posts_df = (
    spark.read
    .format("xml")
    .option("rowTag", "row")
    .load(posts_path)
)

languages_df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .csv(languages_path)
)

In [5]:
# Структура
posts_df.printSchema()
languages_df.printSchema()

posts_df.show(5, truncate=False)
languages_df.show(10, truncate=False)

root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: timestamp (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: timestamp (nullable = true)
 |-- _CreationDate: timestamp (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: timestamp (nullable = true)
 |-- _LastEditDate: timestamp (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)

root
 |-- name: string (nullable = true)
 |-- wikipedia_url: string (nullable

In [7]:
# Очистка списка языков
languages_clean_df = (
    languages_df
    .select(F.lower(F.trim(F.col("name"))).alias("language"))
    .filter(F.col("language").isNotNull())
    .filter(F.col("language") != "")
    .dropDuplicates()
)

languages_clean_df.show(20, truncate=False)

+-----------------------------------+
|language                           |
+-----------------------------------+
|ceylon                             |
|hope                               |
|m#                                 |
|metafont                           |
|mortran                            |
|pl-11                              |
|q (equational programming language)|
|bash and                           |
|objectlogo                         |
|reia                               |
|sail                               |
|comal                              |
|euler                              |
|karel                              |
|lithe                              |
|nwscript                           |
|xmos architecture )                |
|axum                               |
|common lisp (also known as cl)     |
|falcon                             |
+-----------------------------------+
only showing top 20 rows


In [15]:
# Подготовка постов
questions_df = (
    posts_df
    .filter(F.col("_PostTypeId") == 1)
    .filter(F.col("_CreationDate").isNotNull())
    .filter(F.col("_Tags").isNotNull())
    .withColumn("year", F.year(F.to_timestamp("_CreationDate")))
    .filter((F.col("year") >= 2010) & (F.col("year") <= 2020))
    .withColumn("tags_clean", F.lower(F.col("_Tags")))
    .withColumn("tags_clean", F.regexp_replace("tags_clean", "^<|>$", ""))
    .withColumn("tags_array", F.split(F.col("tags_clean"), "><"))
)

questions_df.select("_CreationDate", "year", "_Tags", "tags_array").show(50, truncate=False)

+-----------------------+----+---------------------------------------------------------------+---------------------------------------------------------------+
|_CreationDate          |year|_Tags                                                          |tags_array                                                     |
+-----------------------+----+---------------------------------------------------------------+---------------------------------------------------------------+
|2010-09-22 10:33:21.79 |2010|<c++><character-encoding>                                      |[c++, character-encoding]                                      |
|2010-09-23 06:47:28.92 |2010|<sharepoint><infopath>                                         |[sharepoint, infopath]                                         |
|2010-09-23 08:53:14.017|2010|<iphone><app-store><in-app-purchase>                           |[iphone, app-store, in-app-purchase]                           |
|2010-09-23 11:47:00.833|2010|<symfony1><schem

In [16]:
# Разворачиваем теги в отдельные строки
post_tags_df = (
    questions_df
    .select("year", F.explode("tags_array").alias("tag"))
    .filter(F.col("tag").isNotNull())
    .filter(F.col("tag") != "")
)

post_tags_df.show(20, truncate=False)

+----+------------------+
|year|tag               |
+----+------------------+
|2010|c++               |
|2010|character-encoding|
|2010|sharepoint        |
|2010|infopath          |
|2010|iphone            |
|2010|app-store         |
|2010|in-app-purchase   |
|2010|symfony1          |
|2010|schema            |
|2010|doctrine          |
|2010|fixtures          |
|2010|java              |
|2010|visual-studio-2010|
|2010|stylecop          |
|2010|cakephp           |
|2010|file-upload       |
|2010|swfupload         |
|2010|git               |
|2010|cygwin            |
|2010|putty             |
+----+------------------+
only showing top 20 rows


Теперь каждая строка - это один тег одного вопроса в конкретном году.

In [17]:
# Оставляем только языки программирования
language_mentions_df = (
    post_tags_df
    .join(
        languages_clean_df,
        post_tags_df.tag == languages_clean_df.language,
        how="inner"
    )
    .select(
        post_tags_df.year.alias("year"),
        languages_clean_df.language.alias("language")
    )
)

In [18]:
# Считаем популярность по годам
year_language_counts_df = (
    language_mentions_df
    .groupBy("year", "language")
    .count()
    .withColumnRenamed("count", "mentions")
)

year_language_counts_df.orderBy("year", F.desc("mentions")).show(50, truncate=False)

+----+------------+--------+
|year|language    |mentions|
+----+------------+--------+
|2010|java        |52      |
|2010|php         |46      |
|2010|javascript  |44      |
|2010|python      |26      |
|2010|objective-c |23      |
|2010|c           |20      |
|2010|ruby        |12      |
|2010|delphi      |8       |
|2010|applescript |3       |
|2010|r           |3       |
|2010|perl        |3       |
|2010|bash        |3       |
|2010|haskell     |2       |
|2010|f#          |2       |
|2010|sed         |2       |
|2010|matlab      |1       |
|2010|mouse       |1       |
|2010|xpath       |1       |
|2010|racket      |1       |
|2010|ksh         |1       |
|2010|powershell  |1       |
|2010|ocaml       |1       |
|2010|go          |1       |
|2010|scheme      |1       |
|2010|actionscript|1       |
|2010|dbase       |1       |
|2010|scala       |1       |
|2010|basic       |1       |
|2011|php         |102     |
|2011|java        |93      |
|2011|javascript  |83      |
|2011|python  

In [19]:
# Формируем топ 10 по каждому году
window_spec = Window.partitionBy("year").orderBy(F.desc("mentions"), F.asc("language"))

top10_by_year_df = (
    year_language_counts_df
    .withColumn("rank", F.row_number().over(window_spec))
    .filter(F.col("rank") <= 10)
    .orderBy("year", "rank")
)

top10_by_year_df.show(200, truncate=False)

+----+-----------+--------+----+
|year|language   |mentions|rank|
+----+-----------+--------+----+
|2010|java       |52      |1   |
|2010|php        |46      |2   |
|2010|javascript |44      |3   |
|2010|python     |26      |4   |
|2010|objective-c|23      |5   |
|2010|c          |20      |6   |
|2010|ruby       |12      |7   |
|2010|delphi     |8       |8   |
|2010|applescript|3       |9   |
|2010|bash       |3       |10  |
|2011|php        |102     |1   |
|2011|java       |93      |2   |
|2011|javascript |83      |3   |
|2011|python     |37      |4   |
|2011|objective-c|34      |5   |
|2011|c          |24      |6   |
|2011|ruby       |20      |7   |
|2011|perl       |9       |8   |
|2011|delphi     |8       |9   |
|2011|bash       |7       |10  |
|2012|php        |154     |1   |
|2012|javascript |132     |2   |
|2012|java       |124     |3   |
|2012|python     |69      |4   |
|2012|objective-c|45      |5   |
|2012|c          |27      |6   |
|2012|ruby       |27      |7   |
|2012|bash

In [20]:
# Сохраняем в Parquet
output_path = "programming_languages_top10_2010_2020.parquet"

(
    top10_by_year_df
    .write
    .mode("overwrite")
    .parquet(output_path)
)

In [21]:
# Проверка результата
result_df = spark.read.parquet(output_path)
result_df.orderBy("year", "rank").show(200, truncate=False)

+----+-----------+--------+----+
|year|language   |mentions|rank|
+----+-----------+--------+----+
|2010|java       |52      |1   |
|2010|php        |46      |2   |
|2010|javascript |44      |3   |
|2010|python     |26      |4   |
|2010|objective-c|23      |5   |
|2010|c          |20      |6   |
|2010|ruby       |12      |7   |
|2010|delphi     |8       |8   |
|2010|applescript|3       |9   |
|2010|bash       |3       |10  |
|2011|php        |102     |1   |
|2011|java       |93      |2   |
|2011|javascript |83      |3   |
|2011|python     |37      |4   |
|2011|objective-c|34      |5   |
|2011|c          |24      |6   |
|2011|ruby       |20      |7   |
|2011|perl       |9       |8   |
|2011|delphi     |8       |9   |
|2011|bash       |7       |10  |
|2012|php        |154     |1   |
|2012|javascript |132     |2   |
|2012|java       |124     |3   |
|2012|python     |69      |4   |
|2012|objective-c|45      |5   |
|2012|c          |27      |6   |
|2012|ruby       |27      |7   |
|2012|bash